In [ ]:
from google.colab import drive

print("📁 Mounting Google Drive...")
# Mounts the drive natively; force_remount=True ensures any disconnected sessions are reset
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive connected successfully.")

📁 Mounting Google Drive...


ValueError: mount failed

In [ ]:
print("📦 Installing required library dependencies...")
# Installs streamlit silently without flooding your screen with logs
!pip install -q streamlit
print("✅ Libraries installed successfully.")

In [ ]:
import os
import time

print("🧹 Executing background port cleanup...")
# Force-kill any lingering server or tunneling processes
os.system("pkill -f streamlit")
os.system("pkill -f localtunnel")
os.system("pkill -f cloudflared")

# Clean up temporary log files to prevent reading outdated links
if os.path.exists("streamlit.log"): os.remove("streamlit.log")

print("✅ Environment is clean and ready for deployment.")

In [ ]:
import os

# Create the hidden .streamlit directory structure globally
os.makedirs('/root/.streamlit', exist_ok=True)

theme_config = """
[theme]
primaryColor = "#4A90E2"
backgroundColor = "#0F172A"
secondaryBackgroundColor = "#1E293B"
textColor = "#E2E8F0"
font = "sans serif"
"""

with open('/root/.streamlit/config.toml', 'w') as f:
    f.write(theme_config)

print("🎨 Custom UI clinical theme configuration applied successfully.")

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image

# 1. Page Configuration and Layout
st.set_page_config(
    page_title="PleuraNet AI",
    page_icon="🫁",
    layout="wide",
    initial_sidebar_state="expanded"
)

# 2. Sidebar Status Diagnostics Panel
with st.sidebar:
    st.title("🫁 PleuraNet System")
    st.markdown("---")
    st.success("📡 API Connection: Active")
    st.success("⚡ GPU Acceleration: Ready")
    st.success("🧬 Ensemble Fusion: 50/50 Loaded")
    st.markdown("---")
    st.info("System Model Versions:\n- EfficientNet-B3\n- ConvNeXt")

# 3. Main Title Section
st.title("Clinical Inference Engine")
st.markdown("##### *Upload a patient's Chest X-Ray to generate an automated AI diagnostic consensus report.*")
st.markdown("---")

# 4. Version-Safe Cache-Optimized Model Loader
@st.cache_resource
def load_ai_models():
    # compile=False completely bypasses TensorFlow version object mismatch errors
    model_eff = tf.keras.models.load_model('/content/drive/MyDrive/best_efficientnet_B3_nih.keras', compile=False)
    model_cx = tf.keras.models.load_model('/content/drive/MyDrive/best_convnext_nih.keras', compile=False)

    return model_eff, model_cx

with st.spinner("Initializing Dual-Expert AI Specialists from secure drive..."):
    model_eff, model_cx = load_ai_models()

# 5. Core Interface Columns
col1, col2 = st.columns([1, 1.2], gap="large")

with col1:
    st.subheader("1. Patient Scan Input")
    uploaded_file = st.file_uploader("Select X-Ray File", type=["png", "jpg", "jpeg"])

    if uploaded_file is not None:
        image = Image.open(uploaded_file).convert('RGB')
        st.image(image, caption="Uploaded X-Ray Scan", use_container_width=True)

with col2:
    st.subheader("2. Diagnostics Report")

    if uploaded_file is not None:
        with st.spinner("Running Real-Time Dual-Expert Analysis..."):
            img_np = np.array(image, dtype=np.float32)

            # Preprocessing: Parallel multi-stream resizing
            eff_img = np.expand_dims(cv2.resize(img_np, (300, 300)), axis=0)
            cx_img = np.expand_dims(cv2.resize(img_np, (224, 224)), axis=0)

            # Inference Executions
            p_eff = float(model_eff.predict(eff_img, verbose=0)[0][0] * 100)
            p_cx = float(model_cx.predict(cx_img, verbose=0)[0][0] * 100)

            st.markdown("#### AI Specialist Confidence")

            mc1, mc2 = st.columns(2)
            with mc1:
                st.metric(label="Specialist 1 (Texture)", value=f"{p_eff:.1f}%")
                st.progress(int(p_eff) / 100)
            with mc2:
                st.metric(label="Specialist 2 (Structure)", value=f"{p_cx:.1f}%")
                st.progress(int(p_cx) / 100)

            st.markdown("---")

            # Mathematical Decision Fusion
            p_ens = (0.5 * p_eff) + (0.5 * p_cx)
            variance = abs(p_eff - p_cx)

            st.markdown(f"#### Final Ensemble Consensus: **{p_ens:.1f}%**")

            # Clinical Triage Logic Engine
            if variance > 30.0:
                st.warning(
                    f"**BORDERLINE / SUSPICIOUS**\n\n"
                    f"High variance detected between AI Specialists (Δ = {variance:.1f}%). "
                    f"Automated diagnosis has been suspended. Manual radiologist review is strictly mandated."
                )
            elif p_ens >= 60.0:
                st.error(
                    f"**EFFUSION DETECTED**\n\n"
                    f"Ensemble cross-validation confirms fluid signature presence with high baseline certainty. "
                    f"Immediate clinical verification recommended."
                )
            else:
                # Using standard commas in st.success automatically handles the markdown newlines perfectly
                st.success(
                    "**NORMAL (NO FINDINGS)**\n\n"
                    "Ensemble consensus is well below warning thresholds. "
                    "Lungs appear radiographically clear. Standard routine follow-up advised."
                )
    else:
        st.info("Awaiting file upload to trigger automated diagnosis pipelines.")

In [ ]:
import os
import time
from google.colab import output

print("Starting native local Streamlit background server...")
# Boot the server silently and disable strict security mechanisms for the local proxy
os.system("streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false &>/dev/null &")

# Give the neural engines and server exactly 3 seconds to complete initial blueprint caching
time.sleep(3)

print("\n" + "="*65)
print("✅ DEMO LINK GENERATED SUCCESSFULLY!")
print("Click the link below to open your diagnostic application natively:")
print("="*65 + "\n")

# Call Google Colab's native, zero-block kernel port mapping window
output.serve_kernel_port_as_window(8501)